# Notebook 05: Leakage-Safe Time-Series Feature Engineering
### RetailIQ — Demand Forecasting & Multi-Tool Business Assistant

**Objective:** Engineer calendar, store, promotional, lag, and rolling statistics features strictly on historical shifted targets (`shift(1)`) to guarantee zero target leakage.


In [1]:
import sys, os
from pathlib import Path
# Add project root to path for src imports
project_root = str(Path(os.path.abspath('')).resolve())
if not os.path.exists(os.path.join(project_root, 'src')):
    project_root = str(Path(os.path.abspath('')).resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import pandas as pd
from src.features.forecasting_features import FeatureEngineer

df = pd.read_parquet("data/processed/integrated_sales.parquet")
feat_df = FeatureEngineer.create_features(df, is_training=True)

feature_cols = FeatureEngineer.get_feature_columns()
print(f"Generated {len(feature_cols)} features for {len(feat_df):,} rows.")
print("Feature columns:", feature_cols[:15], "...")


Generated 44 features for 421,570 rows.
Feature columns: ['store_id', 'dept_id', 'store_type_encoded', 'store_size', 'year', 'quarter', 'month', 'week_of_year', 'day_of_year', 'is_holiday', 'is_month_start', 'is_month_end', 'temperature', 'fuel_price', 'cpi'] ...


### Verifying Zero Target Leakage
We assert that `roll_mean_4` at step $t$ is computed strictly from $\{y_{t-1}, y_{t-2}, y_{t-3}, y_{t-4}\}$ and does not contain $y_t$.


In [2]:
# Verify for Store 1, Dept 1
s1d1 = feat_df[(feat_df.store_id == 1) & (feat_df.dept_id == 1)].sort_values("date").reset_index(drop=True)
print("First 6 rows of Store 1, Dept 1:")
s1d1[["date", "weekly_sales", "lag_1", "lag_2", "roll_mean_4", "sales_to_roll_mean_4"]].head(6)


First 6 rows of Store 1, Dept 1:


,date,weekly_sales,lag_1,lag_2,roll_mean_4,sales_to_roll_mean_4
0,2010-02-05,24924.50,24924.50,24924.50,24924.500000,1.000000
1,2010-02-12,46039.49,24924.50,24924.50,24924.500000,0.999960
2,2010-02-19,41595.55,46039.49,24924.50,35481.995000,1.297509
3,2010-02-26,19403.54,41595.55,46039.49,37519.846667,1.108598
4,2010-03-05,21827.90,19403.54,41595.55,32990.770000,0.588133
5,2010-03-12,21043.39,21827.90,19403.54,32216.620000,0.677514


**Conclusion:**
- Zero target leakage verified: `lag_1` strictly matches previous week sales, and `roll_mean_4` is computed on prior shifted sales.
